# Club Player Stats Comparison

Compare skater and goalie statistics for multiple clubs for a selected season and game type.
Enter one or more club abbreviations (e.g. EDM), select a season and game type, then click 'Fetch clubs stats'.

In [ ]:
#!/usr/bin/env python3
import requests
import pandas as pd
from ipywidgets import widgets, VBox, HBox
from IPython.display import display, HTML

# Base server url
BASE_URL = "http://localhost:8080"

# Container for club abbrev widgets
club_widgets = []
seasons = []
club_container = VBox()
club_output = widgets.Output()

def create_club_widget(idx):
    return widgets.Text(description=f'Club {idx}:', style={'description_width': '90px'})

def add_club_widget():
    idx = len(club_widgets) + 1
    w = create_club_widget(idx)
    club_widgets.append(w)
    _update_club_container()

def remove_club_widget(index):
    if 0 <= index < len(club_widgets):
        club_widgets.pop(index)
        _update_club_container()

def _update_club_container():
    children = []
    for i, w in enumerate(club_widgets):
        if i == 0:
            btn = widgets.Button(description='+', button_style='info', layout={'width':'40px'})
            btn.on_click(lambda _: add_club_widget())
            children.append(HBox([w, btn]))
        else:
            btn = widgets.Button(description='−', button_style='danger', layout={'width':'40px'})
            btn.on_click(lambda _, idx=i: remove_club_widget(idx))
            children.append(HBox([w, btn]))
    club_container.children = children

def update_seasons():
    """
    Generate seasons from 1970-1971 up to 2024-2025 (inclusive)
    """
    for y in range(1970, 2025):
        seasons.append(f"{y}{y+1:04d}")

# Initialize with two club inputs
add_club_widget()
add_club_widget()
update_seasons()

season_dropdown = widgets.Dropdown(options=[(s, s) for s in seasons], description='Season:', style={'description_width': '120px'})
game_type_id = widgets.Dropdown(options=[('Regular Season', 2), ('Playoffs', 3)], value=2, description='Game Type:', style={'description_width': '120px'})

In [9]:
# Fetching and display logic
per_club_output = widgets.Output()
agg_output = widgets.Output()

fetch_button = widgets.Button(description='Fetch clubs stats', button_style='success')

def get_club_stats(abbrev, season, game_type_id):
    if not abbrev:
        return None
    url = f"{BASE_URL}/club/{abbrev}/season/{season}/stats"
    try:
        r = requests.get(url, params={'game_type_id': game_type_id})
        if r.status_code == 200:
            return r.json()
        else:
            print(f"Error fetching {abbrev}: {r.status_code}")
    except Exception as e:
        print(f"Request error for {abbrev}: {e}")
    return None

def _player_name(p):
    fn = p.get('firstName', {}).get('default') if isinstance(p.get('firstName'), dict) else (p.get('firstName').default if p.get('firstName') else '')
    ln = p.get('lastName', {}).get('default') if isinstance(p.get('lastName'), dict) else (p.get('lastName').default if p.get('lastName') else '')
    return f"{fn} {ln}".strip()

def display_club_tables(abbrev, club_stats):
    with per_club_output:
        print(f"--- {abbrev} | Season {club_stats.get('season')} | GameType {club_stats.get('gameType')} ---")
        # Skaters
        skaters = club_stats.get('skaters', []) if club_stats else []
        if skaters:
            sk_rows = []
            for s in skaters:
                sk_rows.append({
                    'Player': _player_name(s),
                    'Pos': s.get('positionCode'),
                    'GP': s.get('gamesPlayed'),
                    'G': s.get('goals'),
                    'A': s.get('assists'),
                    'P': s.get('points'),
                    '+/-': s.get('plusMinus'),
                    'PIM': s.get('penaltyMinutes'),
                    'PPG': s.get('powerPlayGoals'),
                    'SHG': s.get('shorthandedGoals'),
                    'GWG': s.get('gameWinningGoals'),
                    'OTG': s.get('overtimeGoals'),
                    'S': s.get('shots'),
                    'S%': s.get('shootingPctg'),
                })
            df_sk = pd.DataFrame(sk_rows)
            display(df_sk)
        else:
            print('No skater stats')
        # Goalies
        goalies = club_stats.get('goalies', []) if club_stats else []
        if goalies:
            gk_rows = []
            for g in goalies:
                gk_rows.append({
                    'Player': _player_name(g),
                    'GP': g.get('gamesPlayed'),
                    'GS': g.get('gamesStarted'),
                    'W': g.get('wins'),
                    'L': g.get('losses'),
                    'OTL': g.get('overtimeLosses'),
                    'GAA': g.get('goalsAgainstAverage'),
                    'SV%': g.get('savePercentage'),
                    'SA': g.get('shotsAgainst'),
                    'Saves': g.get('saves'),
                    'GA': g.get('goalsAgainst'),
                    'SO': g.get('shutouts'),
                })
            df_gk = pd.DataFrame(gk_rows)
            display(df_gk)
        else:
            print('No goalie stats')

def highlight_max_red(row):
    styles = [''] * len(row)
    try:
        values = {}
        for i, col in enumerate(row.index):
            try:
                val = pd.to_numeric(row[col], errors='coerce')
                if pd.notna(val):
                    values[i] = val
            except:
                pass
        if values:
            max_idx = max(values, key=values.get)
            styles[max_idx] = 'color: red; font-weight: bold'
    except Exception:
        pass
    return styles

def aggregate_and_display(all_club_stats):
    # all_club_stats: dict abbrev -> club_stats json
    sk_fields = ['GP','G','A','P','PIM','PPG','SHG','GWG','S']
    agg_rows = []
    # Build aggregated values per club (sum of skaters)
    for abbrev, stats in all_club_stats.items():
        skaters = stats.get('skaters', []) if stats else []
        sums = {f: 0 for f in sk_fields}
        for s in skaters:
            sums['GP'] += s.get('gamesPlayed') or 0
            sums['G'] += s.get('goals') or 0
            sums['A'] += s.get('assists') or 0
            sums['P'] += s.get('points') or 0
            sums['PIM'] += s.get('penaltyMinutes') or 0
            sums['PPG'] += s.get('powerPlayGoals') or 0
            sums['SHG'] += s.get('shorthandedGoals') or 0
            sums['GWG'] += s.get('gameWinningGoals') or 0
            sums['S'] += s.get('shots') or 0
        agg_rows.append({'Club': abbrev, **sums})
    if not agg_rows:
        with agg_output:
            print('No data to aggregate')
        return
    df_agg = pd.DataFrame(agg_rows).set_index('Club').T
    styled = df_agg.style.apply(highlight_max_red, axis=1)
    with agg_output:
        display(HTML('<h3>Aggregated club skater totals</h3>'))
        display(styled)

def on_fetch_click(_):
    per_club_output.clear_output()
    agg_output.clear_output()
    all_club_stats = {}
    with per_club_output:
        for w in club_widgets:
            abbrev = w.value.strip().upper()
            if not abbrev:
                continue
            print(f'Fetching {abbrev}...')
            stats = get_club_stats(abbrev, season_dropdown.value, game_type_id.value)
            if stats:
                all_club_stats[abbrev] = stats
                display_club_tables(abbrev, stats)
            else:
                print(f'No data for {abbrev}')
    # Aggregate and display
    aggregate_and_display(all_club_stats)

fetch_button.on_click(on_fetch_click)
display(VBox([HBox([season_dropdown, game_type_id]), club_container, club_output, fetch_button, per_club_output, agg_output]))